In [1]:
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
pd.options.display.float_format = '{:,.0f}'.format

## Xét promotion trên toàn bộ doanh thu từ trước giờ

In [2]:
query = """
    SELECT 
        oi.order_id,
        oi.product_id,
        oi.quantity,
        oi.unit_price,
        oi.discount_amount,
        p.cogs,

        -- flag có promo
        (oi.promo_id IS NOT NULL) AS has_promo,

        -- revenue
        oi.quantity * oi.unit_price AS revenue,

        -- cost
        oi.quantity * p.cogs AS cost,

        -- profit
        (oi.quantity * oi.unit_price) - (oi.quantity * p.cogs) AS profit,

        -- có bán lỗ
        (oi.unit_price < p.cogs) AS sell_loss_flag

    FROM '../../../data/transaction/order_items.csv' oi
    JOIN '../../../data/master/products.csv' p
    ON oi.product_id = p.product_id
"""

In [3]:
df = duckdb.connect().execute(query).df()
df

,order_id,product_id,quantity,unit_price,discount_amount,cogs,has_promo,revenue,cost,profit,sell_loss_flag
0,1,2400,7,"1,138",0,"1,054",False,"7,968","7,377",591,False
1,2,609,7,"10,166",0,"8,988",False,"71,164","62,914","8,250",False
2,3,396,3,"11,220",0,"10,091",False,"33,661","30,273","3,388",False
3,4,635,5,"10,639",0,"9,205",False,"53,196","46,027","7,169",False
4,6,1935,1,"1,598",0,"1,049",False,"1,598","1,049",549,False
...,...,...,...,...,...,...,...,...,...,...,...
714664,552451,734,4,"3,503","2,802","3,726",True,"14,012","14,903",-891,True
714665,552452,733,1,"3,453",691,"2,285",True,"3,453","2,285","1,168",False
714666,552453,715,4,"1,507","1,206","1,708",True,"6,028","6,831",-803,True
714667,552453,716,3,"1,442",865,"1,763",True,"4,325","5,289",-963,True


In [4]:
df.groupby("has_promo")[["revenue", "cost", "profit"]].mean()

,revenue,cost,profit
has_promo,,,
False,"25,083","20,075","5,007"
True,"19,671","19,410",261


In [5]:
df.groupby("has_promo")["profit"].sum()

has_promo
False   2,195,015,386
True       72,010,681
Name: profit, dtype: float64

In [6]:
df.groupby("has_promo").size()

has_promo
False    438353
True     276316
dtype: int64

In [11]:
print("Tỷ lệ sử dụng promotion = ", 276316/(438353+276316)*100, "%")
print("Tỷ lệ doanh thu từ promotion = ", 72010681/(72010681+2195015386)*100, "%")

Tỷ lệ sử dụng promotion =  38.663493169565214 %
Tỷ lệ doanh thu từ promotion =  3.1764381560593677 %


Xét chung toàn bộ doanh thu từ trước giờ, tỷ lệ sử dụng promotion là `38.66%` và tỷ lệ doanh thu từ promotion là `3.17%`. Điều này cho thấy rằng mặc dù có một phần lớn khách hàng sử dụng promotion, nhưng doanh thu từ promotion chỉ chiếm một phần nhỏ trong tổng doanh thu.


=> Promotion có thể thu hút khách hàng nhưng không phải lúc nào cũng dẫn đến doanh thu cao. Có thể cần xem xét lại chiến lược promotion để tăng hiệu quả và doanh thu từ các chương trình khuyến mãi.

## Xét promotion trên doanh thu qua các năm

In [40]:
query_by_year = """
    SELECT 
        -- flag có promo
        (oi.promo_id IS NOT NULL) AS has_promo,

        -- revenue
        SUM(oi.quantity * oi.unit_price) AS sum_revenue,

        -- cost
        SUM(oi.quantity * p.cogs) AS sum_cost,

        -- profit
        SUM(oi.quantity * oi.unit_price) - SUM(oi.quantity * p.cogs) AS sum_profit,

        --năm
        EXTRACT(year FROM o.order_date) AS order_year

    FROM '../../../data/transaction/order_items.csv' oi
    JOIN '../../../data/master/products.csv' p
    ON oi.product_id = p.product_id
    JOIN '../../../data/transaction/orders.csv' o
    ON oi.order_id = o.order_id
    GROUP BY order_year, has_promo
    ORDER BY order_year, has_promo
"""

In [41]:
df_by_year = duckdb.connect().execute(query_by_year).df()
df_by_year 

,has_promo,sum_revenue,sum_cost,sum_profit,order_year
0,False,"741,497,748","587,461,924","154,035,824",2012
1,False,"995,286,167","791,789,325","203,496,842",2013
2,True,"661,883,250","674,190,778","-12,307,528",2013
3,False,"1,295,140,818","1,032,269,455","262,871,363",2014
4,True,"576,705,064","542,338,001","34,367,063",2014
5,False,"1,145,356,023","912,126,772","233,229,251",2015
6,True,"744,577,804","753,315,046","-8,737,242",2015
7,False,"1,483,054,466","1,193,058,620","289,995,846",2016
8,True,"621,586,211","587,500,773","34,085,439",2016
9,False,"1,178,094,638","944,827,938","233,266,700",2017


Có các năm profit bị âm được thấy từ `time_basic`, là các năm lẻ: 2013,2015,2017,2019,2021

Ở đây thấy rằng trong các năm đó bị âm là do promotion 

Cần xem lại chiến lược promotion trong các năm đó 